# Benchmark: how big a problem this PC can train

How long the fixed RL layer (`core/fixed/`) takes per environment step at the
market's problem size, and, with the market's own cost per step from the clean
branch, how many hours a training run of a given length would take here.

**The environment.** `MarketShapedEnv` (`core/envs/shapes.py`): each agent
observes 42 numbers and sends 4 actions, as a MARLEM market agent; episodes of 24
steps (one day, hourly). The observations are random and the reward is cheap, so
the time measured is RLlib and the networks alone. Grouped paradigms see all
agents at once:

| | DTDE (one module per agent) | CTCE and CTDE (one module for the group) |
|---|---|---|
| Module input | 42 | 42 · N |
| Module output | 4 | 4 · N |

**What one benchmark measures** (`core/fixed/benchmark.py`):

1. **Build**: the networks, optimisers and sampler.
2. **Warm-up** (not timed): iterations until the learner has made its first
   update. PPO: one batch. SAC: `num_steps_sampled_before_learning_starts`
   environment steps, lowered here so learning starts almost at once. APPO: until
   its learner first reports, which is late (it reports only now and then).
3. **Timed**: whole training iterations (sample, then learn) until
   `TIMED_ENV_STEPS` environment steps: seconds per 1000 environment steps and
   gradient steps.
4. **Sampling alone**: the same sampler acting with the current policy, no
   learning. The difference to 3 is the learning.

Each benchmark runs alone, in a fresh process (no other runs sharing the CPU, as
the probe runs did). Results go to `runs/benchmark/` (one JSON per benchmark),
and a finished one is not run again unless `RETIME = True`.

In [ ]:
import json
import multiprocessing
import os
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

# RL root; imports use its parent so market core/ stays separate.
ROOT = next(p / "RL" for p in [Path.cwd(), *Path.cwd().parents] if (p / "RL" / "core").is_dir())
sys.path.insert(0, str(ROOT.parent))
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, [str(ROOT.parent), os.environ.get("PYTHONPATH", "")]))  # the benchmark processes import RL.core too

import numpy as np
import pandas as pd
import torch

from RL.core.envs.shapes import MarketShapedEnv
from RL.core.fixed.benchmark import Bench, benchmark

pd.set_option("display.precision", 2)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("torch", torch.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      "| CPU threads:", torch.get_num_threads())

## a. Choose what to benchmark

- `ALGORITHMS`, `PARADIGMS`, `AGENTS` (N), `DEVICES` (`"cpu"`, `"gpu"`: where the
  learner trains; sampling is always on the CPU).
- `PRESETS`: named settings per algorithm. Every key is passed to the algorithm
  as it is; section b shows what each one means in gradient steps.

**Why the PPO batch size is fixed at 1000 here.** PPO's cost per environment step
does not depend on the batch size: every sampled step is trained on
`num_epochs` times, in minibatches of `minibatch_size`, so a batch of B steps
costs `num_epochs · B / minibatch_size` gradient steps, and per 1000 environment
steps that is `num_epochs · 1000 / minibatch_size` for any B. A smaller batch
only makes each timed iteration shorter. The two presets differ where the cost
is: `"paper"` keeps the paper's minibatch 128 and 30 epochs (about 234 gradient
steps per 1000 environment steps), `"light"` uses minibatch 512 and 10 epochs
(about 20).

**SAC and APPO** are here for comparison; the next experiments use PPO. SAC's
`"quick"` preset starts learning after 256 steps (its default is 1500) and keeps
RLlib's default replay ratio (about one gradient step of 256 samples per
environment step).

In [ ]:
ALGORITHMS = ["ppo"]
PARADIGMS = ["ctce", "ctde", "dtde"]
AGENTS = [5, 13, 39, 118]
DEVICES = ["cpu", "gpu"]
PARTNER_PREFERENCE = False  # True: 42 + N observations per agent
EPISODE_LENGTH = 24
TIMED_ENV_STEPS = 1000  # at least one whole iteration is timed
SAMPLE_ENV_STEPS = 500
MAX_SECONDS = 600
RETIME = False  # True: run finished benchmarks again

PRESETS = {
    "ppo": {
        "paper": {"train_batch_size_per_learner": 1000, "minibatch_size": 128,
                  "num_epochs": 30},
        "light": {"train_batch_size_per_learner": 1000, "minibatch_size": 512,
                  "num_epochs": 10},
    },
    "appo": {
        "default": {"train_batch_size_per_learner": 500},
    },
    "sac": {
        "quick": {"num_steps_sampled_before_learning_starts": 256},
    },
}
OUT = ROOT / "runs" / "benchmark"
MARKET_STEP_CSV = ROOT.parent / "notebooks" / "results" / "market_step.csv"

## b. What each preset means

Computed from the settings, before running anything: steps the learner waits for
before its first update, and gradient steps per 1000 environment steps per
module (DTDE has N modules, each taking these steps on its own agent's data).
SAC's and APPO's gradient steps depend on how RLlib paces them; they are measured
in section d.

In [ ]:
def preset_table() -> pd.DataFrame:
    rows = []
    for algorithm in ALGORITHMS:
        for name, s in PRESETS[algorithm].items():
            row = {"algorithm": algorithm, "preset": name} | s
            if algorithm == "ppo":
                row["warm-up env steps"] = s["train_batch_size_per_learner"]
                row["gradient steps / 1k env steps"] = (
                    s["num_epochs"] * 1000 / s["minibatch_size"]
                )
            elif algorithm == "sac":
                row["warm-up env steps"] = s["num_steps_sampled_before_learning_starts"]
            rows.append(row)
    return pd.DataFrame(rows).set_index(["algorithm", "preset"])


preset_table()

## c. Run

Smallest problems first, so the table fills from the cheap end. The time of each
benchmark is printed as it finishes; the larger DTDE cases take several minutes
each (one module, one optimiser and one update per agent).

In [ ]:
def name(algorithm, paradigm, n, preset, device) -> str:
    return f"{algorithm}_{paradigm}_n{n}_{preset}_{device}"


def plan() -> list[Bench]:
    benches = []
    for n in AGENTS:
        for algorithm in ALGORITHMS:
            for preset, training in PRESETS[algorithm].items():
                for paradigm in PARADIGMS:
                    for device in DEVICES:
                        benches.append(Bench(
                            algorithm,
                            paradigm,
                            MarketShapedEnv,
                            {"n_agents": n, "episode_length": EPISODE_LENGTH,
                             "partner_preference": PARTNER_PREFERENCE},
                            training=training,
                            num_gpus=int(device == "gpu"),
                            timed_env_steps=TIMED_ENV_STEPS,
                            sample_env_steps=SAMPLE_ENV_STEPS,
                            max_seconds=MAX_SECONDS,
                            out=OUT / f"{name(algorithm, paradigm, n, preset, device)}.json",
                        ))
    return benches


def finished(bench: Bench) -> bool:
    path = Path(bench.out)
    return path.exists() and json.loads(path.read_text())["status"] == "completed"


def run_all(benches: list[Bench]) -> None:
    pending = [b for b in benches if RETIME or not finished(b)]
    print(f"{len(benches) - len(pending)} finished, {len(pending)} to run")
    context = multiprocessing.get_context("spawn")
    for bench in pending:
        start = time.time()
        # A fresh process per benchmark: nothing left over from the one before.
        with ProcessPoolExecutor(1, mp_context=context, max_tasks_per_child=1) as pool:
            try:
                record = pool.submit(benchmark, bench).result()
                note = record["status"]
                if record["status"] == "completed":
                    note = f"{record['results']['s_per_1k_env_steps']:7.1f} s / 1k steps"
                else:
                    note += " " + record.get("error", "")[:120]
            except Exception as error:
                note = f"process failed: {error!r}"[:160]
        print(f"{Path(bench.out).stem:32s} {time.time() - start:6.0f} s   {note}",
              flush=True)


BENCHES = plan()
run_all(BENCHES)

## d. Results

One row per benchmark. Seconds per 1000 environment steps, split into sampling
(acting, including RLlib's per-step overhead) and learning; gradient steps
counted from the samples the learner trained on; memory of the process and, on
the GPU, the learner's peak.

In [ ]:
def load() -> pd.DataFrame:
    rows = []
    for bench in BENCHES:
        path = Path(bench.out)
        if not path.exists():
            continue
        r = json.loads(path.read_text())
        row = {
            "algorithm": r["algorithm"],
            "preset": path.stem.split("_")[3],
            "paradigm": r["paradigm"],
            "N": r["n_agents"],
            "device": r["device"],
            "status": r["status"],
        }
        if r["status"] == "completed":
            res = r["results"]
            row |= {
                "modules": r["n_modules"],
                "module in": r["module_obs_dim"],
                "module out": r["module_act_dim"],
                "parameters (M)": r["parameters"] / 1e6,
                "build s": r["build_s"],
                "warm-up steps": r["warmup"]["env_steps"],
                "warm-up s": r["warmup"]["seconds"],
                "s / 1k steps": res["s_per_1k_env_steps"],
                "sampling s / 1k": res["sample_s_per_1k"],
                "learning s / 1k": res["learn_s_per_1k"],
                "grad steps / s": res["gradient_steps_per_s"],
                "grad steps / 1k steps": res["gradient_steps_per_1k_env_steps"],
                "h / 1M steps": res["h_per_1M_env_steps"],
                "RAM GB": res["rss_gb"],
                "GPU GB": res["gpu_peak_gb"],
            }
        else:
            row["error"] = r.get("error", "")
        rows.append(row)
    return pd.DataFrame(rows)


results = load()
results

### CPU or GPU, per problem

Hours per million environment steps of the RL layer alone; the faster device per
row is the one to train on.

In [ ]:
done = results[results.status == "completed"]
by_device = done.pivot_table(
    index=["algorithm", "preset", "paradigm", "N"], columns="device",
    values="h / 1M steps",
)
if {"cpu", "gpu"} <= set(by_device.columns):
    by_device["faster"] = np.where(by_device.gpu < by_device.cpu, "gpu", "cpu")
by_device

## e. With the market: hours per million environment steps

The RL layer's time (the faster device) plus the market's time per step, from
`notebooks/benchmark_market_step.ipynb` on the clean branch
(`MARKET_STEP_CSV`). The market was timed on SimBench networks of 13, 39, 53, 93
and 118 agents; other N are interpolated (and held at the ends). Without that
file, `MARKET_MS` below can be filled in by hand.

This is one environment sampled in the driver process. RLlib can sample several
copies in parallel (`num_env_runners`), which divides the market's part by up to
the number of runners, memory allowing; not measured here.

In [ ]:
MARKET_MS = None  # e.g. {13: 30, 39: 40, 118: 61}, if the CSV is not there

if MARKET_MS is None and MARKET_STEP_CSV.exists():
    market = pd.read_csv(MARKET_STEP_CSV)
    market = market[market.partner_preference == PARTNER_PREFERENCE]
    market = market.groupby("agents").total_ms.mean()
    MARKET_MS = {n: float(np.interp(n, market.index, market.values)) for n in AGENTS}
print("market ms per step:", MARKET_MS)

best = done.loc[done.groupby(["algorithm", "preset", "paradigm", "N"])
                ["h / 1M steps"].idxmin()]
estimate = best[["algorithm", "preset", "paradigm", "N", "device", "h / 1M steps"]]
estimate = estimate.rename(columns={"h / 1M steps": "RL h / 1M"})
if MARKET_MS:
    estimate["market h / 1M"] = estimate.N.map(MARKET_MS) * 1e6 / 1000 / 3600
    estimate["total h / 1M"] = estimate["RL h / 1M"] + estimate["market h / 1M"]
    estimate["steps per night (10 h)"] = (10 / estimate["total h / 1M"] * 1e6).round(-3)
estimate.set_index(["algorithm", "preset", "paradigm", "N"])

## f. How to read it

- **What it does not say: how many steps a problem needs.** That is the real
  unknown, and it grows with N; only learning runs on the market answer it. This
  benchmark turns a number of steps into hours.
- **The cost of the PPO settings.** `"paper"` against `"light"` shows how much of
  the time is the 30 epochs over minibatches of 128.
- **Where the time goes.** Large `sampling` is RLlib's per-step overhead (it
  grows with the number of modules in DTDE); large `learning` is gradient steps,
  which the GPU or lighter settings help with.